# 04 · FinOps: spend, forecast, budgets, approvals
Prices in `data/mccp/prices.json` are **ILLUSTRATIVE**. Budget status: ok < 80% · warn 80–100% · breach ≥ 100%.
Actions: none · notify owner · approve increase · throttle · kill. The standard **$40/user/month** limit maps to the gateway user token quota.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "mccp").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

RUN_LIVE = False  # True only after deploying infra/mccp and filling .env (see docs/mccp/PLAN.md)

from mccp.config import Settings
from mccp.report import Console
settings = Settings.from_env(mode="azure" if RUN_LIVE else "local")
c = Console(color=False)
print("mode:", settings.mode, "(SIMULATED)" if settings.is_local else "(LIVE)")

In [ ]:
from mccp.finops import PriceTable, evaluate, limits_for, load_budgets, load_usage
from mccp.registry import build_inventory
prices = PriceTable.load(settings.price_table_path)
agents, _ = build_inventory(settings)
budgets = load_budgets(settings.data_dir / "budgets.json")
usage = load_usage(settings.data_dir / "usage_fixture.jsonl")
print("$40 per user ->", f"{prices.usd_to_tokens(40):,}", "tokens (gateway quota)")

In [ ]:
for scope in ("agent", "user", "cost_center"):
    views = evaluate(usage, prices, limits_for(scope, agents, budgets), scope, settings.demo_date)
    print(f"--- {scope}")
    for v in views:
        b = v.budget
        print(f"{b.key:<36} ${b.spend_mtd_usd:>8.2f} / ${b.monthly_limit_usd:>6.0f} {b.pct_used:5.1f}% fc ${b.forecast_usd:>8.2f} {b.status:<6} {b.recommended_action}")

## Approval for more (named value `quota-tier2-users`)

In [ ]:
from mccp.demo import build_local_runtime
from mccp.killswitch import approve_quota_increase
from mccp.receipts import ReceiptWriter
rt = build_local_runtime(settings)
receipt = approve_quota_increase(settings, ReceiptWriter(settings.receipts_dir / "notebooks"), "j.chen@contoso-health.example",
                                 authorize=False, approver="k.ito@contoso-health.example", simulator=rt.simulator)
print(receipt.status, receipt.request["url"])
c.json(receipt.request["body"])

## Live spend from Log Analytics (RUN_LIVE)

In [ ]:
if RUN_LIVE:
    from mccp.finops import attach_models, query_log_analytics
    live = attach_models(query_log_analytics(settings), {a.agent_id: a for a in agents})
    print(len(live), "rows")
else:
    print("RUN_LIVE=False: using fixture")